In [0]:
create or replace temporary view mpsii_treatment_table as
SELECT *
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31';

In [0]:
select count(distinct patient_id) from mpsii_treatment_table

In [0]:
create or replace temporary view mpsii_diagnosis_table as 
with mpsii_1dx_specified as (
  SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
mpsii_2dx_specified as (
  select *
  from mpsii_1dx_specified
  where patient_id in (select a.patient_id from mpsii_1dx_specified as a group by a.patient_id having count(distinct a.fill_date) >= 2)
),
mpsii_1dx_unspecified as (
  SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
mpsii_2dx_unspecified as (
  select *
  from mpsii_1dx_unspecified
  where patient_id in (select distinct a.patient_id from mpsii_1dx_unspecified as a group by a.patient_id having count(distinct a.fill_date) >= 2) 
),
mpsii_2dx_specified_tx as (
  select *
  from mpsii_treatment_table where patient_id in (select distinct a.patient_id from mpsii_2dx_specified as a)
),
incremental_patient as (
  select distinct patient_id
  from mpsii_2dx_unspecified
  where patient_id in (select distinct a.patient_id from mpsii_treatment_table as a where a.code in ('54092070001','540920700','J1743'))
  and patient_id not in (select distinct b.patient_id from mpsii_2dx_specified_tx as b)
),
all_dx_patients_claims as (
  select * from mpsii_2dx_specified
  union
  select * from mpsii_1dx_unspecified where patient_id in (select distinct a.patient_id from incremental_patient as a) 
)
select distinct patient_id
from all_dx_patients_claims

In [0]:
create or replace temporary view mpsii_treatment_table_all as
SELECT *
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
        --  WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
          --  WHERE NDC11 IN ('54092070001','540920700')
           where TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
          --  WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31';

In [0]:
-- select count(distinct patient_id) from mpsii_diagnosis_table

-- select count(distinct patient_id)
-- from mpsii_treatment_table_all where patient_id in (select distinct patient_id from mpsii_diagnosis_table) and patient_id not in (
--   select distinct patient_id from mpsii_treatment_table
-- )

select distinct patient_id from mpsii_treatment_table
where patient_id in (select distinct patient_id from mpsii_diagnosis_table)